# Labor reallocation under Shock
## Data
### Data resource
The data is sourced from Compustat global, using Compustat North America as complements. By merging the two datasets, create a cross-country panel data.

To make the data comparable across country, use exchange rate to convert the currency as US dollar. 
* Exchange rate: World Bank Group, Official Exchange Rate (LCU / $), Period Average [Link text](https://data.worldbank.org/indicator/PA.NUS.FCRF)
* GDP deflator: FED- Gross Domestic Product: Implicit Price Deflator (GDPDEF) (2017 = 100) 

The result data: country_panel_deflator&exchange_rate.dta, containing firm-level data, deflator and exchange rate.
This data is ready for the TFP calculation

## TFP Calculation
After cleaning the duplicates, the data is ready for TFP calculation. The TFP calculation will mainly implement the ACF methods, and the package used is acfest. 
- Interpolate for missing employee: only for short gaps (gap ≤ 3); leave as missing for gaps longer than 3
- Gross Investment: the change in ppent plus depreciation
- Using gross investment and PIM to get capital stock
- Intermediate input: cost of goods sold + administrative expense - labor cost
- Value-added output: sale minus intermediate input

In [14]:
use "/Users/2941628y/Library/CloudStorage/OneDrive-UniversityofGlasgow/Desktop/Python/Data_Files/country_panel_TFP_Calculation.dta", clear

*** Interpolate for missing employee ***
egen non_missing_count = rownonmiss(cogs intan ppent sale xlr xsga at dfxa am)
drop if non_missing_count == 0
drop non_missing_count

tsset gvkey fyear
bysort gvkey (fyear): gen miss_group = sum(!missing(emp))
bysort gvkey miss_group: gen gap_length = _N if missing(emp)
bysort gvkey: ipolate emp fyear, gen(emp_interp)

replace emp_interp = . if gap_length > 3 & !missing(gap_length)


*** Currency conversion ***
local vars cogs intan ppent sale xlr xsga at dfxa am 
foreach v of local vars {
  gen `v'_usd = `v' / exchange_rate
  }


*** Real value conversion ***
local vars cogs_usd intan_usd ppent_usd sale_usd xlr_usd xsga_usd at_usd dfxa_usd am_usd
foreach v of local vars {
    gen `v'_real = `v' * (100/gdp_deflator)
}


*** Gross investment ***
xtset gvkey fyear
sort gvkey fyear

* calculate gross investment:
gen GI_tangible = ppent_usd_real[_n] - ppent_usd_real[_n-1] + dfxa_usd_real[_n]

* calculate capital stock: 
gen k = ppent_usd_real[_n-1] * (1-0.08) + GI_tangible[_n]
replace k = 0 if k < 0

* log of capital: 
replace k = log(k)


*** Intermeidate input ***
gen var = cogs_usd_real + xsga_usd_real - xlr_usd_real

* log of material: 
qui gen double m = log(var)


*** Labor input ***
qui gen double l = log(emp_interp)


*** Value-added output *** 
gen output = sale - var
drop if output <= 0 
drop if missing(output)
qui gen double va = log(va)


*** ACF ***
acfest va, state(k) proxy(m) free(l) i(gvkey) t(fyear) nbs(200) va
predict tfp_acf_k, omega


*** Winsorizing ***
winsor2 tfp_acf_k, gen(tfp_acf_k1) p(1)





(40,051 observations deleted)



Panel variable: gvkey (unbalanced)
 Time variable: fyear, 1986 to 2024, but with gaps
         Delta: 1 unit


(447,821 missing values generated)

(332227 missing values generated)

(32,981 real changes made, 32,981 to missing)


(131,906 missing values generated)
(66,729 missing values generated)
(49,340 missing values generated)
(124,316 missing values generated)
(596,351 missing values generated)
(154,960 missing values generated)
(644 missing values generated)
(215,315 missing values generated)
(257,074 missing values generated)


(131,906 missing values generated)
(66,729 missing values generated)
(49,340 missing values generated)
(124,316 missing values generated)
(596,351 missing values generated)
(154,960 missing values generated)
(644 missing values generated)
(215,315 missing values generated)
(257,074 missing values generated)


Panel variable: gvkey (unbalanced)
 Time variable: fyear, 1986 to 2024, but with gaps
         Delta: 1 unit


(

r(198);
r(198);


In [16]:
sum tfp_acf_k


    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
   tfp_acf_k |     80,041    4.102772      1.3718  -6.923678   20.33511


Winsorizing at 1-99%:
Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
 tfp_acf_k_1 |     80,041    4.100474    1.292464   .7453176   7.936516

Winsorizing at 5-95%:
Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
 tfp_acf_k_5 |     80,041    4.089869    1.137896   1.982724   6.334499